# 📖 Notebook 4: Service Discovery

In a modern backend you rarely have just "one payments server." You have 5 — and
they scale up and down, crash, and get replaced. Your **order service** needs to
know *which instances of `payments` are alive right now* so it can send requests
to a healthy one.

That problem is called **service discovery**. ZooKeeper solves it elegantly with
**ephemeral child nodes** under a well-known path.

| Approach | Method | Problem |
|----------|--------|---------|
| 🔴 Bad | Hardcoded IPs in config | Every scale-up/crash needs a redeploy |
| 🟡 Better | Shared config file + polling | Slow to detect dead instances, stale lists |
| 🟢 Best | ZooKeeper ephemeral children + watch | Instant registration + instant crash detection |

## Learning Objectives

By the end of this notebook, you'll understand:

- Why "just configure the IPs" breaks in a cloud environment
- How services can **register themselves** at startup
- How clients get a **live list** of healthy instances that updates automatically
- How crashed instances are removed from the list *with no extra code*


## 🛠️ Setup

Start the ZooKeeper ensemble first:

```bash
cd 03-technologies/coordination/zookeeper
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear: `Cmd+Shift+P` → "Reload Window".


In [ ]:
import json
import random
import threading
import time


---
## 🔴 Bad: Hardcoded IPs in a Config File

The simplest (worst) approach: every client has a list of backend IPs baked into
its configuration.

```python
PAYMENT_SERVERS = [
    "10.0.0.11:8080",
    "10.0.0.12:8080",
    "10.0.0.13:8080",
]
```

Problems:

- **Scale-up requires a code change / redeploy.** Add a 4th server? Update the list
  everywhere.
- **Crashes aren't detected.** The client keeps sending requests to a dead IP.
- **No health info.** Even if the process is alive, it might be overloaded.


In [ ]:
# Simulate: clients call a random backend from a hardcoded list
HARDCODED_BACKENDS = ["10.0.0.11:8080", "10.0.0.12:8080", "10.0.0.13:8080"]

# Pretend one of them crashed
dead = "10.0.0.12:8080"

def send_request(backends):
    chosen = random.choice(backends)
    if chosen == dead:
        return f"  ❌ Timeout talking to {chosen} (crashed, but client didn't know)"
    return f"  ✅ 200 OK from {chosen}"

print("Client sending 5 requests using a hardcoded backend list:")
for _ in range(5):
    print(send_request(HARDCODED_BACKENDS))
print()
print("The client keeps trying the crashed server because the config is static.")


---
## 🟡 Better: Shared Config File + Polling

Put the backend list in a central place (a file on S3, a database row, etc.) and
have clients re-read it every N seconds.

- ✅ You can add/remove backends without redeploying clients.
- ❌ Whoever runs `deploy.sh` must remember to **update the list**.
- ❌ If a server **crashes silently**, it stays in the list until someone notices.
- ❌ Clients always have a stale view — anywhere from 0 to N seconds old.


In [ ]:
# Simulate a shared "registry" everyone reads from
shared_registry = {
    "payments": ["10.0.0.11:8080", "10.0.0.12:8080", "10.0.0.13:8080"],
}

# An ops engineer forgot to remove a crashed node from the file
# -> clients will keep trying it
dead = "10.0.0.12:8080"

def client_poll_and_call():
    backends = shared_registry["payments"]  # re-read every call
    chosen = random.choice(backends)
    if chosen == dead:
        return f"  ❌ still hitting the dead {chosen}"
    return f"  ✅ 200 OK from {chosen}"

for _ in range(5):
    print(client_poll_and_call())

print()
print("A shared config is better than hardcoding, but it doesn't know who's alive.")


---
## 🟢 Best: ZooKeeper Ephemeral Children + ChildrenWatch

The classic ZooKeeper pattern for service discovery is two moves:

1. **Each service instance registers itself** by creating an **ephemeral** child
   node under `/services/<service-name>/`, with its host:port (and any metadata)
   as the node's data.
2. **Clients set a `ChildrenWatch`** on `/services/<service-name>/`. Any time
   a child is added or removed, the watch fires and the client re-reads the
   list.

```
/services
└── /services/payments
    ├── /services/payments/instance-0001   ← ephemeral, data = "10.0.0.11:8080"
    ├── /services/payments/instance-0002   ← ephemeral, data = "10.0.0.12:8080"
    └── /services/payments/instance-0003   ← ephemeral, data = "10.0.0.13:8080"
```

Why this works:

- **Self-registration**: no human has to maintain a list.
- **Crash = automatic deregistration**: the ephemeral node disappears when the
  process dies, because its ZK session expires.
- **Live updates**: clients find out in milliseconds, not minutes.


In [ ]:
from kazoo.client import KazooClient

zk = KazooClient(hosts="localhost:2181,localhost:2182,localhost:2183")
zk.start()
print("Connected to ZooKeeper!")


In [ ]:
class ServiceInstance:
    """A backend service that registers itself with ZooKeeper on startup."""

    def __init__(self, service_name, host, port, zk_hosts):
        self.service_name = service_name
        self.host = host
        self.port = port
        self.zk = KazooClient(hosts=zk_hosts)
        self.zk.start()
        self.node_path = None

    def register(self):
        base = f"/services/{self.service_name}"
        self.zk.ensure_path(base)

        # Node data: JSON so we can include version, weight, region, etc.
        metadata = json.dumps({
            "host": self.host,
            "port": self.port,
            "started_at": time.time(),
        }).encode()

        # EPHEMERAL + SEQUENCE -> auto-removed on crash, unique name per instance
        self.node_path = self.zk.create(
            f"{base}/instance-",
            value=metadata,
            ephemeral=True,
            sequence=True,
        )
        print(f"  🟢 {self.service_name} instance registered as {self.node_path}")

    def shutdown(self, crash=False):
        # A real shutdown would delete the node; a crash just drops the session
        if not crash and self.node_path and self.zk.exists(self.node_path):
            self.zk.delete(self.node_path)
        self.zk.stop()


In [ ]:
# Start 3 "payments" service instances
instances = [
    ServiceInstance("payments", "10.0.0.11", 8080, "localhost:2181,localhost:2182,localhost:2183"),
    ServiceInstance("payments", "10.0.0.12", 8080, "localhost:2181,localhost:2182,localhost:2183"),
    ServiceInstance("payments", "10.0.0.13", 8080, "localhost:2181,localhost:2182,localhost:2183"),
]
for inst in instances:
    inst.register()


In [ ]:
class ServiceDiscoveryClient:
    """A client that keeps a live list of healthy backends using a ChildrenWatch."""

    def __init__(self, service_name, zk_hosts):
        self.service_name = service_name
        self.base = f"/services/{service_name}"
        self.zk = KazooClient(hosts=zk_hosts)
        self.zk.start()
        self.zk.ensure_path(self.base)
        self.backends = []
        self._start_watch()

    def _start_watch(self):
        @self.zk.ChildrenWatch(self.base)
        def handle_children(children):
            # Called every time a child is added or removed under self.base
            new_list = []
            for child in children:
                try:
                    data, _ = self.zk.get(f"{self.base}/{child}")
                    new_list.append(json.loads(data.decode()))
                except Exception:
                    pass  # child might have vanished between get_children and get
            self.backends = new_list
            addrs = [f"{b['host']}:{b['port']}" for b in new_list]
            print(f"  🔔 client: live backends -> {addrs}")

    def pick_backend(self):
        if not self.backends:
            return None
        b = random.choice(self.backends)
        return f"{b['host']}:{b['port']}"

# Start a discovery client (in a real app this runs inside the caller service)
client = ServiceDiscoveryClient("payments", "localhost:2181,localhost:2182,localhost:2183")
time.sleep(1)
print()
print(f"Client will route to: {client.pick_backend()}")


### 💥 Crash Demo: dead instances vanish from the list automatically

Let's simulate one of the payment servers crashing. We expect the client's
live-backend list to shrink **without writing any extra code**.


In [ ]:
print("Before crash:")
print(f"  instances visible to client: {len(client.backends)}")
print()

print("💥 Simulating crash of instance[1] (connection drop)...")
instances[1].shutdown(crash=True)

# Ephemeral nodes disappear when the session expires.
# By default, kazoo uses a ~10s session timeout, so we wait a few seconds.
time.sleep(6)

print()
print("After crash:")
print(f"  instances visible to client: {len(client.backends)}")
print()
print("✅ The client's list updated automatically. No redeploy, no config edit,")
print("   no health-check service — just ephemeral nodes + a watch.")


### 🧠 Why This Pattern Is So Popular

This exact pattern — "register as an ephemeral child, watch the parent" — is the
skeleton of many real systems:

- **Kafka (pre-KRaft)** used ZooKeeper to track which brokers were alive. Each
  broker registered as an ephemeral znode under `/brokers/ids`.
- **HBase** uses ZooKeeper to track region servers and the active master.
- **Solr, Druid, Pinot** use ZooKeeper for cluster membership and coordination.

(Many of these systems are slowly moving off ZooKeeper — for example Kafka's
**KRaft** mode replaces ZooKeeper with an internal Raft-based controller quorum —
but the pattern itself is canonical.)


---
## 📊 Summary: Bad → Better → Best

| | 🔴 Hardcoded IPs | 🟡 Shared config + polling | 🟢 ZooKeeper ephemeral children |
|---|---|---|---|
| **Add a new instance** | Redeploy all clients | Edit shared file | Instance self-registers |
| **Detect a crash** | Never (until a human notices) | After manual edit | Automatic (session expiry) |
| **Update latency** | Minutes-hours | Seconds-minutes (poll interval) | Milliseconds (watch) |
| **Client code needed** | List of IPs | Polling loop | One `ChildrenWatch` |

### Key Takeaway

Service discovery is just two ZooKeeper primitives working together:

- **Ephemeral nodes** → *"I'm alive as long as my node exists."*
- **Watches** → *"Tell me when the membership changes."*

### When to Use This

- You have multiple replicas of a service behind a logical name.
- Instances come and go (autoscaling, container restarts).
- You want clients to react in milliseconds instead of minutes.

Tools like Consul, Eureka, and Kubernetes Services do the same thing with
different building blocks — but understanding this ZooKeeper version teaches you
what's happening underneath.


In [ ]:
# Cleanup
for inst in instances:
    try:
        inst.shutdown()
    except Exception:
        pass

try:
    client.zk.stop()
except Exception:
    pass

if zk.exists("/services"):
    zk.delete("/services", recursive=True)
zk.stop()
print("Cleaned up ZooKeeper nodes. Done!")
